In [1]:
# ============================================================
# NOTEBOOK: 03_excel_data_prep.ipynb
# PROJECT : UPI Transaction Fraud & Risk Analytics
# PURPOSE : Merge cleaned CSVs into 1 file for Excel Dashboard
# ============================================================

import pandas as pd
import os

# ── Step 1: Load all 5 cleaned CSVs ──────────────────────────
users_df     = pd.read_csv('data/cleaned/users_clean.csv')
merchants_df = pd.read_csv('data/cleaned/merchants_clean.csv')
txns_df      = pd.read_csv('data/cleaned/transactions_clean.csv')
fraud_df     = pd.read_csv('data/cleaned/fraud_labels_clean.csv')
failures_df  = pd.read_csv('data/cleaned/payment_failures_clean.csv')

print(" All 5 CSVs loaded!")
print(f"  Users        : {users_df.shape}")
print(f"  Merchants    : {merchants_df.shape}")
print(f"  Transactions : {txns_df.shape}")
print(f"  Fraud Labels : {fraud_df.shape}")
print(f"  Failures     : {failures_df.shape}")


# ── Step 2: Merge into one Excel-ready dataset ───────────────
excel_data = (
    txns_df
    .merge(fraud_df, on='txn_id', how='left')
    .merge(users_df[['user_id', 'name', 'user_city', 'age_group',
                      'kyc_status', 'account_age_days',
                      'new_account_flag', 'account_cohort']],
           on='user_id', how='left')
    .merge(merchants_df[['merchant_id', 'merchant_name', 'category',
                          'merchant_city', 'is_verified',
                          'merchant_fraud_rate']],
           on='merchant_id', how='left')
    .merge(failures_df, on='txn_id', how='left')
)

print(f"\n Merged Excel dataset shape: {excel_data.shape}")
print(f"Columns: {list(excel_data.columns)}")


# ── Step 3: Fill nulls from the merge ────────────────────────
excel_data['failure_reason'] = excel_data['failure_reason'].fillna('N/A')
excel_data['retry_count']    = excel_data['retry_count'].fillna(0)
excel_data['resolved']       = excel_data['resolved'].fillna('N/A')
excel_data['fraud_type']     = excel_data['fraud_type'].fillna('No Fraud')

print("\n Nulls handled!")
print(excel_data.isnull().sum().sum(), "total nulls remaining")


# ── Step 4: Export final Excel-ready CSV ─────────────────────
os.makedirs('data/excel', exist_ok=True)

excel_data.to_csv('data/excel/excel_dashboard_data.csv', index=False)

print("\n Final file exported!")
print("Location: data/excel/excel_dashboard_data.csv")
print(f"Final shape: {excel_data.shape}")

 All 5 CSVs loaded!
  Users        : (5000, 9)
  Merchants    : (500, 7)
  Transactions : (50000, 16)
  Fraud Labels : (50000, 3)
  Failures     : (3517, 4)

 Merged Excel dataset shape: (50000, 33)
Columns: ['txn_id', 'user_id', 'merchant_id', 'amount', 'timestamp', 'status', 'channel', 'device_type', 'txn_hour', 'txn_day', 'txn_month', 'txn_month_name', 'late_night_flag', 'amount_zscore', 'high_amount_flag', 'amount_bucket', 'is_fraud', 'fraud_type', 'name', 'user_city', 'age_group', 'kyc_status', 'account_age_days', 'new_account_flag', 'account_cohort', 'merchant_name', 'category', 'merchant_city', 'is_verified', 'merchant_fraud_rate', 'failure_reason', 'retry_count', 'resolved']

 Nulls handled!
0 total nulls remaining

 Final file exported!
Location: data/excel/excel_dashboard_data.csv
Final shape: (50000, 33)
